<a href="https://colab.research.google.com/github/muhammadanaswork/DevelopersHub-AI-Internship/blob/main/Phase2_Task_2_ML_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 2: End-to-End ML Pipeline with Scikit-learn Pipeline API
**Objective:** Build a reusable and production-ready machine learning pipeline for predicting customer churn.
**Goal:** Load the Telco Churn Dataset, implement data preprocessing using `Pipeline`, train a Random Forest model, optimize it using `GridSearchCV`, and export the final pipeline using `joblib`.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

# Load Telco Churn Dataset from a reliable raw GitHub source
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
print("Loading Telco Customer Churn dataset...")
df = pd.read_csv(url)

# Basic cleanup: 'TotalCharges' is sometimes a blank string, let's fix that
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True) # Drop the few missing values

# Drop the customerID column as it's not useful for prediction
df.drop('customerID', axis=1, inplace=True)

print("Dataset loaded and cleaned successfully!")
display(df.head())

Loading Telco Customer Churn dataset...
Dataset loaded and cleaned successfully!


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### 1. Data Splitting and Preprocessing Pipeline
We will separate our numeric and categorical columns and apply transformations to them simultaneously using `ColumnTransformer` and `Pipeline`.

In [2]:
# Separate Features (X) and Target (y)
X = df.drop('Churn', axis=1)
y = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0) # Convert target to binary

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify categorical and numerical columns
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

# Create preprocessing steps
numeric_transformer = StandardScaler() # Scales numbers to have 0 mean and 1 variance
categorical_transformer = OneHotEncoder(handle_unknown='ignore') # Converts text categories to binary columns

# Combine preprocessing steps into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessing pipeline constructed.")

Preprocessing pipeline constructed.


### 2. Model Development, Training, and Hyperparameter Tuning
We will chain our preprocessor with a `RandomForestClassifier` and use `GridSearchCV` to find the best hyperparameters.

In [3]:
# Create the full pipeline: Preprocessing + Model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Define the hyperparameters we want to tune
# (Keeping it small so it runs fast in Colab)
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth':[5, 10]
}

# Set up GridSearchCV
print("Starting Grid Search for Hyperparameter Tuning (this may take a minute)...")
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1)

# Train the model using the pipeline
grid_search.fit(X_train, y_train)

print(f"Best Parameters Found: {grid_search.best_params_}")

Starting Grid Search for Hyperparameter Tuning (this may take a minute)...
Best Parameters Found: {'classifier__max_depth': 10, 'classifier__n_estimators': 50}


### 3. Evaluation and Exporting for Production
We evaluate our optimized pipeline on the unseen test data and then export it using `joblib` so it can be reused later without retraining.

In [4]:
# Predict using the best model found by GridSearch
y_pred = grid_search.predict(X_test)

# Evaluate metrics
print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Export the complete, tuned pipeline using joblib
model_filename = "churn_prediction_pipeline.joblib"
joblib.dump(grid_search.best_estimator_, model_filename)

print(f"Production-ready pipeline exported successfully as '{model_filename}'")

Accuracy Score: 0.7924662402274343

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.90      0.86      1033
           1       0.65      0.49      0.55       374

    accuracy                           0.79      1407
   macro avg       0.74      0.69      0.71      1407
weighted avg       0.78      0.79      0.78      1407

Production-ready pipeline exported successfully as 'churn_prediction_pipeline.joblib'


### Final Summary / Insights
1. **Pipeline Construction:** By using Scikit-Learn's `Pipeline`, we successfully bundled scaling, encoding, and model predictions into a single, cohesive object. This prevents data leakage during training.
2. **Hyperparameter Tuning:** `GridSearchCV` systematically tested combinations of Random Forest parameters and selected the optimal depth and estimator count.
3. **Production Readiness:** The final pipeline was exported via `joblib`. In a real-world scenario, a backend developer can now easily load this `.joblib` file and run `pipeline.predict(new_customer_data)` without needing to rewrite any preprocessing logic.